# Tugas Take-Home Sesi 05 — Transfer Learning untuk Klasifikasi Citra dengan Data Terbatas
**Dataset:** Oxford-IIIT Pet (subset 10 kelas: 5 kucing + 5 anjing) | **Backbone utama:** ResNet-18 | **Platform:** Google Colab (GPU T4)

Notebook ini menjalankan seluruh eksperimen **B1–B5** secara berurutan, mencatat metrik mentah ke `results/`, lalu menghasilkan tabel (C1), kurva loss (C1), failure analysis (C3), dan Grad-CAM (C5).

> **Cara pakai:** Runtime → Change runtime type → **T4 GPU**, lalu **Runtime → Run all**. Semua angka pada laporan harus diambil dari `results/metrics_summary.csv` / `results/training_logs.csv`.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "pandas", "matplotlib"], check=False)

In [ ]:
import os, random, time, json, copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms, datasets

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

# ---------------- Reproduksibilitas ----------------
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CFG = dict(
    batch_size=32,
    max_epochs=30,
    patience=5,          # early stopping pada validation loss (3-5 sesuai protokol)
    weight_decay=1e-2,
    lr_scratch=1e-3,     # B1: seluruh parameter
    lr_head=1e-3,        # B2/B3: classifier head (fc)
    lr_backbone=1e-4,    # B3: layer4
    num_classes=10,
    seed=SEED,
)

OUT = Path("results"); OUT.mkdir(exist_ok=True)
print("torch", torch.__version__, "| device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU (sangat lambat, aktifkan GPU!)")

## 1. Data: Oxford-IIIT Pet (10 kelas) dengan split stratified 70/15/15
Subset: 5 ras kucing (Abyssinian, Bengal, Persian, Siamese, Sphynx) dan 5 ras anjing (Beagle, Boxer, Pug, Shiba Inu, Samoyed).
Split `trainval` + `test` bawaan digabung terlebih dahulu, lalu dibagi ulang secara stratified (seed 42).

In [ ]:
CAT_BREEDS = ["Abyssinian", "Bengal", "Persian", "Siamese", "Sphynx"]
DOG_BREEDS = ["Beagle", "Boxer", "Pug", "Shiba Inu", "Samoyed"]
SELECTED = CAT_BREEDS + DOG_BREEDS

raw_tv = datasets.OxfordIIITPet(root="data", split="trainval", target_types="category", download=True)
raw_te = datasets.OxfordIIITPet(root="data", split="test", target_types="category", download=True)

all_classes = raw_tv.classes
name_to_idx = {n: i for i, n in enumerate(all_classes)}
missing = [b for b in SELECTED if b not in name_to_idx]
assert not missing, f"Nama kelas tidak ditemukan: {missing}. Cek: {all_classes}"
old2new = {name_to_idx[n]: k for k, n in enumerate(SELECTED)}

paths, labels = [], []
for ds in (raw_tv, raw_te):
    for p, l in zip(ds._images, ds._labels):
        if l in old2new:
            paths.append(str(p)); labels.append(old2new[l])
labels = np.array(labels)

idx = np.arange(len(paths))
tr_idx, tmp_idx = train_test_split(idx, test_size=0.30, stratify=labels, random_state=SEED)
va_idx, te_idx = train_test_split(tmp_idx, test_size=0.50, stratify=labels[tmp_idx], random_state=SEED)

print(f"Total citra: {len(paths)} | train: {len(tr_idx)} | val: {len(va_idx)} | test: {len(te_idx)}")
print(pd.DataFrame({"kelas": SELECTED,
                    "train": np.bincount(labels[tr_idx], minlength=10),
                    "val": np.bincount(labels[va_idx], minlength=10),
                    "test": np.bincount(labels[te_idx], minlength=10)}))

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Augmentasi hanya untuk train set (sesuai starter code)
train_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
# Deterministik: dipakai untuk val/test dan untuk skenario 'tanpa augmentasi'
eval_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class PetSubset(Dataset):
    def __init__(self, indices, transform):
        self.indices, self.transform = indices, transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        j = self.indices[i]
        img = Image.open(paths[j]).convert("RGB")
        return self.transform(img), int(labels[j])

def _worker_init(worker_id):
    s = SEED + worker_id
    np.random.seed(s); random.seed(s)

def make_loader(indices, tfm, shuffle):
    g = torch.Generator(); g.manual_seed(SEED)
    return DataLoader(PetSubset(indices, tfm), batch_size=CFG["batch_size"], shuffle=shuffle,
                      num_workers=2, pin_memory=True, generator=g, worker_init_fn=_worker_init)

## 2. Model, strategi pelatihan, dan fungsi latih/evaluasi
- **B1**: ResNet-18 `weights=None`, semua parameter dilatih (lr 1e-3).
- **B2**: pretrained, backbone beku, hanya `fc` baru dilatih (lr 1e-3).
- **B3**: pretrained, `layer4` + `fc` dilatih dengan *differential learning rate* (1e-4 / 1e-3).
- Lapisan **BatchNorm yang dibekukan dijaga dalam mode `eval()`** agar statistik ImageNet (running mean/var) tidak berubah oleh data kecil.

In [ ]:
def build_model(arch="resnet18", strategy="B2", pretrained=None):
    nc = CFG["num_classes"]
    use_pre = (strategy != "B1") if pretrained is None else pretrained

    if arch == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if use_pre else None)
        m.fc = nn.Linear(m.fc.in_features, nc)
        head, last = m.fc, [m.layer4]
    elif arch == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if use_pre else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, nc)
        head, last = m.classifier, [m.features[7], m.features[8]]
    elif arch == "mobilenet_v3_small":
        m = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if use_pre else None)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, nc)
        head, last = m.classifier, list(m.features[-3:])
    else:
        raise ValueError(arch)

    if strategy in ("B2", "B3"):
        for p in m.parameters():
            p.requires_grad = False
        for p in head.parameters():
            p.requires_grad = True
        if strategy == "B3":
            for blk in last:
                for p in blk.parameters():
                    p.requires_grad = True
    return m, head, last

def count_params(m):
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    total = sum(p.numel() for p in m.parameters())
    return trainable, total

def make_param_groups(m, head, last, strategy):
    if strategy == "B1":
        return [{"params": list(m.parameters()), "lr": CFG["lr_scratch"]}], f"all={CFG['lr_scratch']:g}"
    if strategy == "B2":
        return [{"params": list(head.parameters()), "lr": CFG["lr_head"]}], f"head={CFG['lr_head']:g}"
    backbone = [p for blk in last for p in blk.parameters()]
    return ([{"params": backbone, "lr": CFG["lr_backbone"]},
             {"params": list(head.parameters()), "lr": CFG["lr_head"]}],
            f"last_blocks={CFG['lr_backbone']:g}; head={CFG['lr_head']:g}")

def set_train_mode(m):
    m.train()
    for mod in m.modules():
        if isinstance(mod, nn.modules.batchnorm._BatchNorm):
            ps = list(mod.parameters())
            if ps and not any(p.requires_grad for p in ps):
                mod.eval()   # BN beku -> statistik tetap (tidak ikut update)

def run_epoch(model, loader, criterion, optimizer):
    set_train_mode(model)
    tot_loss = correct = n = 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        out = model(x); loss = criterion(out, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward(); optimizer.step()
        tot_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item(); n += x.size(0)
    return tot_loss / n, correct / n

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    tot_loss = n = 0
    ys, ps, prob = [], [], []
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        out = model(x)
        tot_loss += criterion(out, y).item() * x.size(0); n += x.size(0)
        ys.append(y.cpu()); ps.append(out.argmax(1).cpu()); prob.append(F.softmax(out, 1).cpu())
    y = torch.cat(ys).numpy(); p = torch.cat(ps).numpy(); pr = torch.cat(prob).numpy()
    return dict(loss=tot_loss / n, acc=accuracy_score(y, p), f1=f1_score(y, p, average="macro"),
                y=y, preds=p, probs=pr)

def _sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

RESULTS = {}

def save_logs():
    pd.DataFrame([r["summary"] for r in RESULTS.values()]).to_csv(OUT / "metrics_summary.csv", index=False)
    pd.concat([pd.DataFrame(r["history"]) for r in RESULTS.values()]).to_csv(OUT / "training_logs.csv", index=False)
    with open(OUT / "metrics_raw.json", "w") as f:
        json.dump({"config": CFG, "experiments": {k: {"summary": v["summary"], "history": v["history"]} for k, v in RESULTS.items()}}, f, indent=2)

def fit(name, arch, strategy, aug=True):
    set_seed(SEED)
    model, head, last = build_model(arch, strategy)
    model = model.to(DEVICE)
    trainable, total = count_params(model)
    print(f"\n=== {name} | {arch} | {strategy} | aug={aug} ===")
    print(f"Trainable Parameters: {trainable:,} / {total:,}")
    if arch == "resnet18" and strategy == "B2":
        assert trainable == 5130, "B2 pada ResNet-18 seharusnya 5.130 parameter trainable"

    groups, lr_desc = make_param_groups(model, head, last, strategy)
    optimizer = torch.optim.AdamW(groups, weight_decay=CFG["weight_decay"])
    criterion = nn.CrossEntropyLoss()

    train_loader = make_loader(tr_idx, train_tfm if aug else eval_tfm, shuffle=True)
    val_loader = make_loader(va_idx, eval_tfm, shuffle=False)

    best_val, best_epoch, best_state, wait, hist = float("inf"), 0, None, 0, []
    _sync(); t0 = time.time()
    for ep in range(1, CFG["max_epochs"] + 1):
        tl, ta = run_epoch(model, train_loader, criterion, optimizer)
        vr = evaluate(model, val_loader, criterion)
        hist.append(dict(exp=name, epoch=ep, train_loss=tl, train_acc=ta,
                         val_loss=vr["loss"], val_acc=vr["acc"], val_macro_f1=vr["f1"]))
        print(f"ep {ep:02d} | train {tl:.4f}/{ta:.3f} | val {vr['loss']:.4f}/{vr['acc']:.3f}")
        if vr["loss"] < best_val - 1e-4:
            best_val, best_epoch, wait = vr["loss"], ep, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= CFG["patience"]:
                print(f"Early stopping pada epoch {ep} (best epoch = {best_epoch})"); break
    _sync(); duration = time.time() - t0

    model.load_state_dict(best_state)
    val_r = evaluate(model, val_loader, criterion)
    test_r = evaluate(model, make_loader(te_idx, eval_tfm, False), criterion)
    train_clean = evaluate(model, make_loader(tr_idx, eval_tfm, False), criterion)  # tanpa augmentasi

    torch.save({"state_dict": best_state, "arch": arch, "strategy": strategy}, OUT / f"ckpt_{name}.pt")

    summary = dict(
        exp=name, arch=arch, strategy=strategy, augmentation=bool(aug),
        trainable_params=int(trainable), total_params=int(total), lr_groups=lr_desc,
        epochs_run=len(hist), best_epoch=int(best_epoch), train_time_s=round(float(duration), 2),
        train_acc_clean=float(train_clean["acc"]), val_acc=float(val_r["acc"]), val_loss=float(val_r["loss"]),
        val_macro_f1=float(val_r["f1"]), test_acc=float(test_r["acc"]), test_macro_f1=float(test_r["f1"]),
        gap_train_val=float(train_clean["acc"] - val_r["acc"]), seed=SEED,
    )
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in summary.items()})
    RESULTS[name] = dict(summary=summary, history=hist, test=test_r)
    save_logs()
    return RESULTS[name]

## 3. Eksperimen B1–B3 (ResNet-18, dengan augmentasi)

In [ ]:
fit("B1_scratch", "resnet18", "B1", aug=True)
fit("B2_feature_extraction", "resnet18", "B2", aug=True)
fit("B3_progressive_ft", "resnet18", "B3", aug=True)

## 4. B4 — Uji kontrol data augmentation (kondisi B3)
B3 dipilih (ditetapkan sebelum melihat hasil) karena memiliki kapasitas yang dilatih lebih besar sehingga risiko *overfitting* — dan efek augmentasi — lebih terlihat.
Kedua skenario dijalankan terpisah dengan seed identik.

In [ ]:
fit("B4_B3_noaug", "resnet18", "B3", aug=False)
fit("B4_B3_aug", "resnet18", "B3", aug=True)

## 5. B5 (Bonus) — Arsitektur pembanding
Strategi terbaik antara B2 dan B3 (berdasarkan **validation accuracy**, bukan test) dijalankan pada EfficientNet-B0 dan MobileNetV3-Small.

In [ ]:
best_strategy = max(["B2_feature_extraction", "B3_progressive_ft"], key=lambda k: RESULTS[k]["summary"]["val_acc"])
best_strategy = "B2" if best_strategy.startswith("B2") else "B3"
print("Strategi terbaik (val acc):", best_strategy)
fit("B5_efficientnet_b0", "efficientnet_b0", best_strategy, aug=True)
fit("B5_mobilenet_v3_small", "mobilenet_v3_small", best_strategy, aug=True)

## 6. C1 — Tabel konsolidasi dan kurva pembelajaran

In [ ]:
summary = pd.DataFrame([r["summary"] for r in RESULTS.values()])
cols = ["exp", "arch", "strategy", "augmentation", "trainable_params", "lr_groups", "best_epoch", "epochs_run",
        "train_time_s", "train_acc_clean", "val_acc", "test_macro_f1", "gap_train_val"]
display(summary[cols])

def df_to_md(df, floatfmt="{:.4f}"):
    cs = list(df.columns)
    lines = ["| " + " | ".join(cs) + " |", "|" + "|".join(["---"] * len(cs)) + "|"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(floatfmt.format(r[c]) if isinstance(r[c], float) else str(r[c]) for c in cs) + " |")
    return "\n".join(lines)

(OUT / "consolidated_table.md").write_text(df_to_md(summary[cols]))

hist_df = pd.concat([pd.DataFrame(r["history"]) for r in RESULTS.values()])
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)
for ax, name in zip(axes, ["B1_scratch", "B2_feature_extraction", "B3_progressive_ft"]):
    h = hist_df[hist_df.exp == name]
    ax.plot(h.epoch, h.train_loss, marker="o", label="train loss")
    ax.plot(h.epoch, h.val_loss, marker="s", label="val loss")
    ax.axvline(RESULTS[name]["summary"]["best_epoch"], ls="--", c="gray", label="best epoch")
    ax.set_title(name); ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT / "fig_loss_curves_B1_B2_B3.png", dpi=150); plt.show()

# Ringkasan B4: celah train-val
b4 = summary[summary.exp.str.startswith("B4")][["exp", "train_acc_clean", "val_acc", "gap_train_val", "test_macro_f1"]]
display(b4)

## 7. C3 — Failure analysis pada model terbaik (dipilih berdasarkan validation accuracy)

In [ ]:
cands = {k: v for k, v in RESULTS.items() if not k.startswith("B4")}
best_name = max(cands, key=lambda k: cands[k]["summary"]["val_acc"])
print("Model terbaik:", best_name, "| test macro-F1:", round(cands[best_name]["summary"]["test_macro_f1"], 4))

tr = RESULTS[best_name]["test"]
print(classification_report(tr["y"], tr["preds"], target_names=SELECTED, digits=3))

cm = confusion_matrix(tr["y"], tr["preds"])
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues"); plt.colorbar(im)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(SELECTED, rotation=60, ha="right"); ax.set_yticklabels(SELECTED)
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
ax.set_xlabel("Prediksi"); ax.set_ylabel("Ground truth"); ax.set_title(f"Confusion matrix — {best_name}")
plt.tight_layout(); plt.savefig(OUT / "fig_confusion_matrix.png", dpi=150); plt.show()

conf = tr["probs"][np.arange(len(tr["y"])), tr["preds"]]
wrong = np.where(tr["preds"] != tr["y"])[0]
order = wrong[np.argsort(-conf[wrong])]   # kesalahan paling 'yakin' lebih dulu
show = order[:8]
print(f"Jumlah salah prediksi di test: {len(wrong)} dari {len(tr['y'])}")

disp_tfm = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])
fail_rows = []
if len(show):
    ncol = 4; nrow = int(np.ceil(len(show) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 4.4 * nrow)); axes = np.array(axes).reshape(-1)
    for ax in axes: ax.axis("off")
    for ax, i in zip(axes, show):
        pth = paths[te_idx[i]]
        ax.imshow(disp_tfm(Image.open(pth).convert("RGB")))
        gt, pr = SELECTED[tr["y"][i]], SELECTED[tr["preds"][i]]
        ax.set_title(f"GT: {gt}\nPred: {pr} ({conf[i]:.2f})", fontsize=10)
        fail_rows.append(dict(file=pth, ground_truth=gt, prediction=pr, confidence=float(conf[i])))
    plt.tight_layout(); plt.savefig(OUT / "fig_failure_analysis.png", dpi=150); plt.show()
    pd.DataFrame(fail_rows).to_csv(OUT / "failure_samples.csv", index=False)
    display(pd.DataFrame(fail_rows))

**Catatan C3 (isi sendiri di laporan):** setelah melihat gambar di atas, tuliskan hipotesis penyebab tiap kesalahan
(ambiguitas visual antar ras, sudut ekstrem, oklusi, pencahayaan, bias latar belakang, dsb.). Hipotesis harus berdasarkan apa yang benar-benar terlihat pada gambar.

## 8. C5 (Opsional) — Grad-CAM `layer4`: sebelum (B2) vs sesudah (B3) fine-tuning

In [ ]:
def load_model(name):
    ck = torch.load(OUT / f"ckpt_{name}.pt", map_location=DEVICE)
    m, _, _ = build_model(ck["arch"], ck["strategy"], pretrained=False)
    m.load_state_dict(ck["state_dict"])
    return m.to(DEVICE).eval()

def gradcam(model, x, target):
    model.eval()
    store = {}
    h = model.layer4.register_forward_hook(lambda m, i, o: store.__setitem__("a", o))
    x = x.clone().requires_grad_(True)       # agar graf terbentuk walau backbone beku
    out = model(x)
    grad = torch.autograd.grad(out[0, target], store["a"])[0]
    h.remove()
    w = grad.mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((w * store["a"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.detach().cpu().numpy()

m_before, m_after = load_model("B2_feature_extraction"), load_model("B3_progressive_ft")

# 4 gambar: prioritaskan kesalahan model terbaik, lengkapi dengan sampel test acak
rng = np.random.RandomState(SEED)
sel = list(show[:4])
extra = [i for i in rng.permutation(len(te_idx)) if i not in sel]
sel = (sel + extra)[:4]

fig, axes = plt.subplots(3, 4, figsize=(14, 10.5))
for c, i in enumerate(sel):
    img = Image.open(paths[te_idx[i]]).convert("RGB")
    x = eval_tfm(img).unsqueeze(0).to(DEVICE)
    y = int(labels[te_idx[i]]); base = disp_tfm(img)
    axes[0, c].imshow(base); axes[0, c].set_title(f"GT: {SELECTED[y]}", fontsize=10)
    for r, (mdl, tag) in enumerate([(m_before, "B2 (sebelum FT)"), (m_after, "B3 (sesudah FT)")], start=1):
        cam = gradcam(mdl, x, y)
        pred = mdl(x).argmax(1).item()
        axes[r, c].imshow(base); axes[r, c].imshow(cam, cmap="jet", alpha=0.45)
        axes[r, c].set_title(f"{tag}\npred: {SELECTED[pred]}", fontsize=9)
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout(); plt.savefig(OUT / "fig_gradcam_B2_vs_B3.png", dpi=150); plt.show()

## 9. Penyimpanan artefak

In [ ]:
save_logs()
print(sorted(p.name for p in OUT.iterdir()))
# Opsional (Colab): unduh seluruh hasil
# !zip -qr results.zip results && python -c "from google.colab import files; files.download('results.zip')"